# Sales Analyst 1.5B v2 Continual Fine-Tune (QLoRA)

**Purpose**: Incrementally extend the v1 1.5B adapter with v2 API knowledge (`forecast`, `cohort_analysis`).  
**Starts from**: `models/adapters/1.5b/` (v1 checkpoint)  
**Data**: `data/v2/delta.jsonl` (~200 pairs, v2 signal types only)  
**Output**: `models/adapters/1.5b-v2/`  

Only new signal types (`forecast_up`, `cohort_question`) are in the delta set.  
This avoids catastrophic forgetting of v1 behaviour.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q peft==0.10.0 transformers==4.40.2 trl==0.8.6 bitsandbytes==0.43.1 accelerate==0.29.3 datasets==2.18.0 sentencepiece==0.2.0 pyyaml

In [ ]:
import json
from pathlib import Path

import yaml
import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
REPO_ROOT   = Path("/content/drive/MyDrive/salestools-analyst")
CONFIG_PATH = REPO_ROOT / "training/config/lora_1.5b.yaml"
PROMPT_PATH = REPO_ROOT / "training/config/system_prompt.txt"
DELTA_DATA  = REPO_ROOT / "data/v2/delta.jsonl"
V1_ADAPTER  = REPO_ROOT / "models/adapters/1.5b"      # starting checkpoint
ADAPTER_OUT = REPO_ROOT / "models/adapters/1.5b-v2"   # output

ADAPTER_OUT.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

SYSTEM_PROMPT = Path(PROMPT_PATH).read_text().strip()
print("Base model:", cfg["base_model"])
print("Starting adapter:", V1_ADAPTER)

In [ ]:
CHATML_TEMPLATE = (
    "<|im_start|>system\n{system}<|im_end|>\n"
    "<|im_start|>user\n{user}<|im_end|>\n"
    "<|im_start|>assistant\n{assistant}<|im_end|>"
)

records = []
with open(DELTA_DATA) as f:
    for line in f:
        pair = json.loads(line.strip())
        if not pair.get("verified", False):
            continue
        records.append({"text": CHATML_TEMPLATE.format(
            system=SYSTEM_PROMPT,
            user=pair["question"],
            assistant=pair["code"],
        )})

dataset = Dataset.from_list(records)
print(f"Loaded {len(dataset)} v2 delta pairs")

In [ ]:
# Load base model + apply v1 LoRA adapter as starting point
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(V1_ADAPTER),   # load the already-fine-tuned v1 adapter
    max_seq_length=cfg["max_seq_len"],
    dtype=None,
    load_in_4bit=True,
)

# Apply fresh LoRA on top (continual learning — new adapters added)
model = FastLanguageModel.get_peft_model(
    model,
    r=cfg["lora_rank"],
    target_modules=cfg["target_modules"],
    lora_alpha=cfg["lora_alpha"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=cfg["seed"],
)
print("Continual fine-tune adapter applied.")
model.print_trainable_parameters()

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=cfg["max_seq_len"],
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=cfg["batch_size"],
        gradient_accumulation_steps=cfg["gradient_accumulation"],
        warmup_steps=2,
        num_train_epochs=5,    # more epochs — smaller dataset
        learning_rate=1e-4,    # lower LR for continual fine-tune
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=cfg["seed"],
        output_dir=str(ADAPTER_OUT / "checkpoints"),
        save_steps=50,
        save_total_limit=2,
        report_to="none",
    ),
)
print("Trainer ready.")

In [ ]:
trainer_stats = trainer.train()
print(f"Done. Runtime: {trainer_stats.metrics.get('train_runtime', 0):.0f}s")

In [ ]:
model.save_pretrained(str(ADAPTER_OUT))
tokenizer.save_pretrained(str(ADAPTER_OUT))
print(f"v2 adapter saved to {ADAPTER_OUT}")

## Next Steps

1. Download `models/adapters/1.5b-v2/` from Drive
2. Export: `MODEL_SIZE=1.5b-v2 ADAPTER_PATH=models/adapters/1.5b-v2/ bash training/export.sh`
3. Lifecycle eval: `python eval/run_eval.py --model sales-analyst-1.5b-v2 --held-out data/v1/held_out.jsonl`
4. Compare: `python eval/compare.py eval/reports/*1.5b-*.json eval/reports/*1.5b-v2-*.json`